# Notebook Purpose

The purpose of this notebook is to replicate the random forest results.

## Imports

In [1]:
import subprocess
import psutil
import functools
import random
import time
import statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta

import humanize
import GPUtil as GPU
import numpy as np
import torch
import pandas as pd
from tqdm import trange
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from datasets.dataset_handler import DatasetHandler
from datasets.enums import FeatureType, DatasetSource

## Attributes

In [2]:
handler = DatasetHandler()
feature_type = FeatureType.ECFP
dataset_sources = list(DatasetSource)

## Methods

### Retrieve CMD Output

Method to run and retrieve results from the CMD process.

In [3]:
def get_cmd_output(command):
    return subprocess.check_output(
        command,
        stderr=subprocess.STDOUT,
        shell=True
    ).decode('UTF-8')

## Main Logic

### Device Information

In [4]:
cpu = get_cmd_output('cat /proc/cpuinfo | grep -E "model name"')
cpu = cpu.split('\n')[0].split('\t: ')[-1]
physical_cpu_count = psutil.cpu_count(logical=False)
logical_cpu_count = psutil.cpu_count() # physical count X no. of threads per physical core
try:
    cuda_version = get_cmd_output('nvcc --version | grep -E "Build"')
except subprocess.CalledProcessError:
    cuda_version = "Not Available"
try:
    gpu = get_cmd_output("nvidia-smi -L")
except subprocess.CalledProcessError:
    gpu = "Not Available"
general_ram_gb = humanize.naturalsize(psutil.virtual_memory().available)
try:
    gpu_ram_total_mb = GPU.getGPUs()[0].memoryTotal
except IndexError:
    gpu_ram_total_mb = "Not Available"

print(f"CPU: '{cpu}'")
print(f"Physical CPU Count: '{physical_cpu_count}'")
print(f"Logical CPU Count: '{logical_cpu_count}'")
print(f"CUDA Version: '{cuda_version}'")
print(f"GPU: '{gpu}'")
print(f"Available RAM: '{general_ram_gb}'")
print(f"GPU RAM: '{gpu_ram_total_mb}'")

CPU: 'AMD Ryzen 5 3600 6-Core Processor'
Physical CPU Count: '6'
Logical CPU Count: '12'
CUDA Version: 'Not Available'
GPU: 'Not Available'
Available RAM: '23.7 GB'
GPU RAM: 'Not Available'


### Parameters

In [5]:
rounds = 20
randomseed = 12
torch.manual_seed(randomseed) 
np.random.seed(randomseed)
random.seed(randomseed)
torch.cuda.manual_seed(randomseed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.is_available()
torch.backends.cudnn.benchmark = False # selects fastest conv algo
torch.backends.cudnn.deterministic = True
ml_model = "random_forest"

combinations = [
    [10, 10],
    [5, 10], 
    [1, 10], 
    [1, 5], 
    [1, 1]
]

save_columns = [
    'EXPERIMENT_DATE',
    'CPU',
    'CPU COUNT',
    'GPU',
    'GPU RAM',
    'RAM',
    'CUDA',
    'DATASET SOURCE',
    'FEATURE TYPE',
    'MODEL ARCHITECTURE',
    'NUMBER POSITIVE',
    'NUMBER NEGATIVE',
    'TARGET',
    'ACCURACY',
    'ROC',
    'PRC',
    'TRAIN ROC',
    'TRAIN PRC',
    'EPISODES',
    'TRAINING TIME',
    'ROC VALUES',
    'PRC VALUES'
]

### Runs

In [6]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

combinations = [
    [10, 10],
    [5, 10], 
    [1, 10], 
    [1, 5], 
    [1, 1]
]

for dataset_source in dataset_sources:
    # train tasks discarded: RF is a no-transfer baseline, fit per-target on the support set only
    _, test_dfs = handler.load_train_test_set(dataset_source=dataset_source, feature_type=feature_type)
    dataset_source_val = dataset_source.value

    result_df = pd.DataFrame(columns=save_columns) # initialise the results DataFrame
    results_csv = results_dir / f"{ml_model}/{dataset_source_val}.csv"
    results_csv.parent.mkdir(parents=True, exist_ok=True)
    
    if results_csv.exists():
        print(f"Skipping random forest experiment for '{dataset_source_val}' dataset. Results already exist at '{results_csv}'")
        continue

    for no_pos, no_neg in combinations:
        for target in test_dfs.keys():
            dt_run_start = datetime.now(timezone.utc).strftime("%d/%m/%Y %H:%M:%S")
            print(f"\n{dt_run_start} - Running random forest experiment for the '{dataset_source_val}' dataset with '{no_pos}' positives and '{no_neg}' negatives for '{target}' target")
            
            running_roc = []
            running_prc = []

            start_time = time.time()
            for r in trange(rounds):
                test_df = test_dfs[target]

                support_neg = test_df[test_df['y'] == 0].sample(no_neg)
                support_pos = test_df[test_df['y'] == 1].sample(no_pos)

                train_data = pd.concat([support_neg, support_pos])
                test_data = test_df.drop(train_data.index)

                train_X, train_y = list(train_data['mol'].to_numpy()), train_data['y'].to_numpy(dtype=np.int16)
                test_X, test_y = list(test_data['mol'].to_numpy()), test_data['y'].to_numpy(dtype=np.int16)

                model = RandomForestClassifier(n_estimators=100)
                model.fit(train_X, train_y)
                probs_y = model.predict_proba(test_X)

                roc = roc_auc_score(test_y, probs_y[:, 1])
                prc = average_precision_score(test_y, probs_y[:, 1])
                
                running_roc.append(roc)
                running_prc.append(prc)

            end_time = time.time()
            duration = str(timedelta(seconds=(end_time - start_time)))
            
            rounds_roc = f"{statistics.mean(running_roc):.3f} \u00B1 {statistics.stdev(running_roc):.3f}"
            rounds_prc = f"{statistics.mean(running_prc):.3f} \u00B1 {statistics.stdev(running_prc):.3f}"
            rounds_rec = pd.DataFrame([
                [
                    dt_run_start,
                    cpu,
                    logical_cpu_count,
                    gpu,
                    gpu_ram_total_mb,
                    general_ram_gb,
                    cuda_version,
                    dataset_source_val,
                    feature_type.value,
                    ml_model,
                    no_pos,
                    no_neg,
                    target,
                    -1,
                    rounds_roc,
                    rounds_prc,
                    -1,
                    -1,
                    -1,
                    duration,
                    running_roc,
                    running_prc
                ]],
                columns=save_columns
             )
            result_df = pd.concat([result_df, rounds_rec])
    
    print(f"Saving experiment results to CSV at '{results_csv}'")
    result_df.to_csv(results_csv, index=False)

Skipping random forest experiment for 'tox21' dataset. Results already exist at 'results/random_forest/tox21.csv'
Skipping random forest experiment for 'muv' dataset. Results already exist at 'results/random_forest/muv.csv'
Skipping random forest experiment for 'dude_gpcr' dataset. Results already exist at 'results/random_forest/dude_gpcr.csv'
